In [66]:
# import pyspark.sql.functions as F
# print(dir(F))

StatementMeta(, a234de3e-0893-4294-9d50-190b1a27bd6e, 70, Finished, Available, Finished, False)

In [67]:
# Type here in the cell editor to add code!
# Imports all PySpark SQL functions Like: col() to_date() sum() year() etc.
from pyspark.sql.functions import*

StatementMeta(, a234de3e-0893-4294-9d50-190b1a27bd6e, 71, Finished, Available, Finished, False)

In [68]:
# Read the table
df = spark.read.table("ecommerce_bronze.bronze.bronze_amz_sales_raw")

StatementMeta(, a234de3e-0893-4294-9d50-190b1a27bd6e, 72, Finished, Available, Finished, False)

In [69]:
# Print the table
#display(df)
#display(df.orderBy(desc("last_updated")))

StatementMeta(, a234de3e-0893-4294-9d50-190b1a27bd6e, 73, Finished, Available, Finished, False)

In [70]:
#df.count()

StatementMeta(, a234de3e-0893-4294-9d50-190b1a27bd6e, 74, Finished, Available, Finished, False)

In [71]:
df.printSchema()

StatementMeta(, a234de3e-0893-4294-9d50-190b1a27bd6e, 75, Finished, Available, Finished, False)

root
 |-- index: integer (nullable = true)
 |-- Order_ID: string (nullable = true)
 |-- Date: date (nullable = true)
 |-- Status: string (nullable = true)
 |-- Fulfilment: string (nullable = true)
 |-- Sales_Channel: string (nullable = true)
 |-- ship_service_level: string (nullable = true)
 |-- Style: string (nullable = true)
 |-- SKU: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Size: string (nullable = true)
 |-- ASIN: string (nullable = true)
 |-- Courier_Status: string (nullable = true)
 |-- Qty: integer (nullable = true)
 |-- currency: string (nullable = true)
 |-- Amount: double (nullable = true)
 |-- ship_city: string (nullable = true)
 |-- ship_state: string (nullable = true)
 |-- ship_postal_code: double (nullable = true)
 |-- ship_country: string (nullable = true)
 |-- promotion_ids: string (nullable = true)
 |-- B2B: boolean (nullable = true)
 |-- fulfilled_by: string (nullable = true)
 |-- Unnamed_22: string (nullable = true)
 |-- last_updated: tim

### **Step-1 : Function to correct the column name in the table using "snake_case standard"**

In [72]:
# Manually converting the column name to snake_case standard
rename_map = {
    "Order ID"           : "order_id",
    "Date"               : "date",
    "Status"             : "status",
    "Fulfilment"         : "fulfilment",
    "Sales Channel "     : "sales_channel",  
    "ship-service-level" : "ship_service_level",
    "Style"              : "style",
    "SKU"                : "sku",
    "Category"           : "category",
    "Size"               : "size",
    "ASIN"               : "asin",
    "Courier Status"     : "courier_status",
    "Qty"                : "qty",
    "currency"           : "currency",
    "Amount"             : "amount",
    "ship-city"          : "ship_city",
    "ship-state"         : "ship_state",
    "ship-postal-code"   : "ship_postal_code",
    "ship-country"       : "ship_country",
    "promotion-ids"      : "promotion_ids",
    "B2B"                : "customer_type",
    "fulfilled-by"       : "fulfilled_by",
}

for old_name, new_name in rename_map.items():
    if old_name in df.columns:
        df = df.withColumnRenamed(old_name, new_name)

StatementMeta(, a234de3e-0893-4294-9d50-190b1a27bd6e, 76, Finished, Available, Finished, False)

In [73]:
# This an an more professional and logical code alternative of the above code
# def clean_columns(dataframe):    # Define a function that takes a Spark DataFrame as input
#     return dataframe.toDF(*[     # Rename all columns at once using toDF()
#         col.strip().lower().replace(" ", "_")  # For each column: remove extra spaces, convert to lowercase,   # and replace spaces with underscores
#         for col in dataframe.columns   # Loop through all existing column names
#     ])

# df = clean_columns(df) # Call the function and replace df with the cleaned DataFrame
# # printing the updated table
# display(df)

StatementMeta(, a234de3e-0893-4294-9d50-190b1a27bd6e, 77, Finished, Available, Finished, False)

### **Checking which columns we can drop**

In [74]:
#display(df.groupBy("fulfilled_by").count())# this will give an count of all the distinct values saperately. df.count() → total rows groupBy().count() → per category, filter().count() → conditional count
#display(df.groupBy("currency").count()) # same with this gives two count of all the distinct values in the column
#display(df.groupBy("ship_country").count())
# display((df.select('date')).distinct()) # just checking that there are unique dates in the dataset or not if all the dates are same then there is no meaning of this colum

StatementMeta(, a234de3e-0893-4294-9d50-190b1a27bd6e, 78, Finished, Available, Finished, False)

### **Step-2 : Drop unnecessary columns**

In [75]:
df = df.drop("index","sales_channel","currency","ship_country","Unnamed: 22","fulfilled_by")
 # removing sales_channel because there are only 124 values are Non-Amazon and also as of now this column have not much impact on dashboard
 # removing currency because all of it have same currency which is "INR"
 # removing ship_country because all have same values which is "IN"
 # removing unnamed_22 because it is unnecessary

StatementMeta(, a234de3e-0893-4294-9d50-190b1a27bd6e, 79, Finished, Available, Finished, False)

### **Step-3 : Data Type Casting**

In [76]:
#df.printSchema()

StatementMeta(, a234de3e-0893-4294-9d50-190b1a27bd6e, 80, Finished, Available, Finished, False)

In [77]:
# converting the datatypes 
from pyspark.sql.types import IntegerType, DecimalType, BooleanType
from pyspark.sql.functions import to_date, col

df = (
    df
    .withColumn("date", to_date(col("date"), "MM-dd-yy"))
    .withColumn("qty", col("qty").cast(IntegerType()))
    .withColumn("customer_type", col("customer_type").cast(BooleanType()))
    .withColumn("amount", col("amount").cast(DecimalType(18,2)))
)

StatementMeta(, a234de3e-0893-4294-9d50-190b1a27bd6e, 81, Finished, Available, Finished, False)

In [78]:
#df.printSchema()

StatementMeta(, a234de3e-0893-4294-9d50-190b1a27bd6e, 82, Finished, Available, Finished, False)

### **Checks the null values in all the column. It quickly tells you which columns have missing values.**

In [79]:
# columns have null values 
display(df.select([
    sum(
        col(c).isNull().cast("int")
    ).alias(c)
    for c in df.columns
]))
# df.columns returns all column names
# for c in df.columns iterates through each column name
# col(c).isNull() checks if the value in that column is NULL
# if NULL → True, if not NULL → False
# cast("int") converts True → 1 and False → 0
# sum() adds all the 1s to count total NULL values in that column
# alias(c) renames the output column with the original column name

StatementMeta(, a234de3e-0893-4294-9d50-190b1a27bd6e, 83, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 78172136-a717-45e2-ad75-9e26f8111556)

In [80]:
#column have null values and also all the values which are 0 in the column
display(df.select([
    sum(
        (
            col(c).isNull() |
            (col(c).cast("string") == "") |
            (
                (col(c).cast("double").isNotNull()) &
                (col(c).cast("double") == 0)
            )
        ).cast("int")
    ).alias(c)
    for c in df.columns
]))

StatementMeta(, a234de3e-0893-4294-9d50-190b1a27bd6e, 84, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 889dc5e1-4831-4d21-a6de-c3fc1b20be9d)

In [81]:
# code to check the null values 
# display(df.filter((col("amount").isNull()) | (col("amount") == 0)).count())
# display(df.filter(col("order_id")==0))
# df.count() 

StatementMeta(, a234de3e-0893-4294-9d50-190b1a27bd6e, 85, Finished, Available, Finished, False)

### **Checking Duplicate values**

In [82]:
# checking the duplicate values for the order_id
# display(df.groupBy('order_id')
# .count()
# .filter(col('count') > 1)
# .orderBy(col('count').desc())
# .show(10)
# )
# total_orders_distinct = df.select('order_id').distinct().count()
# total_orders = df.select('order_id').count()
# total_dup= total_orders - total_orders_distinct
# print("Total Duplicates = ", total_dup)  

StatementMeta(, a234de3e-0893-4294-9d50-190b1a27bd6e, 86, Finished, Available, Finished, False)

In [83]:
# # Check duplicate Order IDs (same order_id, same SKU)
# dup_orders = df.groupBy("order_id", "sku")\
#     .count()\
#     .filter(col("count") > 1)\
#     .orderBy(col("count").desc())

# print(f"\nDuplicate (order_id + sku) groups : {dup_orders.count()}")
# dup_orders.show()


# There are 3 possible reasons why the same order_id + sku repeats: (It is important because if due to system error if there is an more duplicate record then the sales revenue gets increase Unethically)
# Reason 1 — Genuine purchase 
# Customer intentionally ordered 2 or more of the same item. Perfectly valid — Qty should reflect this, not duplicate rows.
# Reason 2 — Data export bug 
# The CSV export from Amazon duplicated some rows. This directly inflates your revenue KPIs. 
# Reason 3 — System re-processing 
# The order was updated or reprocessed and got logged twice in the system.

StatementMeta(, a234de3e-0893-4294-9d50-190b1a27bd6e, 87, Finished, Available, Finished, False)

In [84]:
#This is also an most important check because if due to system error if there is an more duplicate record then the sales revenue gets increase Unethically
#Most strict — only check records that affect revenue KPIs
df_valid = df.filter(
    (col("status") != "Cancelled") &
    (col("amount") > 0) &
    (col("qty") > 0)
)

dup_orders = df_valid.groupBy("order_id", "sku") \
    .count() \
    .filter(col("count") > 1) \
    .orderBy(col("count").desc())

print(f"\nDuplicate (order_id + sku) groups : {dup_orders.count()}")
dup_orders.show()

StatementMeta(, a234de3e-0893-4294-9d50-190b1a27bd6e, 88, Finished, Available, Finished, False)


Duplicate (order_id + sku) groups : 3
+-------------------+---------------+-----+
|           order_id|            sku|count|
+-------------------+---------------+-----+
|405-8669298-3850736|MEN5025-KR-XXXL|    2|
|407-4853873-4978725|    J0230-SKD-M|    2|
|171-3249942-2207542|SET323-KR-NP-XL|    2|
+-------------------+---------------+-----+



In [85]:
# Check for negative values
# neg_qty    = df.filter(col("qty") < 0).count()
# neg_amount = df.filter(col("amount") < 0).count()

# print(f"Negative Qty records    : {neg_qty}")
# print(f"Negative Amount records : {neg_amount}")

StatementMeta(, a234de3e-0893-4294-9d50-190b1a27bd6e, 89, Finished, Available, Finished, False)

### **Categorical cloumn details**

In [86]:
# This code will give us all the categorical cloumn details of the values in the column
print("\n" + "="*60)
print("CATEGORICAL COLUMN PROFILING")
print("="*60)

categorical_cols = [
    "status", "fulfilment", "category",
    "size", "ship_service_level",
    "courier_status","customer_type"
]

for col_name in categorical_cols:
    print(f"\n── {col_name} ──")
    df.groupBy(col_name) \
      .count() \
      .orderBy("count", ascending=False) \
      .show(truncate=False )

StatementMeta(, a234de3e-0893-4294-9d50-190b1a27bd6e, 90, Finished, Available, Finished, False)


CATEGORICAL COLUMN PROFILING

── status ──
+-----------------------------+-----+
|status                       |count|
+-----------------------------+-----+
|Shipped                      |77804|
|Shipped - Delivered to Buyer |28769|
|Cancelled                    |18332|
|Shipped - Returned to Seller |1953 |
|Shipped - Picked Up          |973  |
|Pending                      |658  |
|Pending - Waiting for Pick Up|281  |
|Shipped - Returning to Seller|145  |
|Shipped - Out for Delivery   |35   |
|Shipped - Rejected by Buyer  |11   |
|Shipping                     |8    |
|Shipped - Lost in Transit    |5    |
|Shipped - Damaged            |1    |
+-----------------------------+-----+


── fulfilment ──
+----------+-----+
|fulfilment|count|
+----------+-----+
|Amazon    |89698|
|Merchant  |39277|
+----------+-----+


── category ──
+-------------+-----+
|category     |count|
+-------------+-----+
|Set          |50284|
|kurta        |49877|
|Western Dress|15500|
|Top          |10622|
|Ethni

In [87]:
# #simply grouping the qty by showing the count. doing this to check weather it makes sense to make an calculated column.
#df.groupBy('qty').count().orderBy("count", ascending=False).show()

StatementMeta(, a234de3e-0893-4294-9d50-190b1a27bd6e, 91, Finished, Available, Finished, False)

### Order Status Mapping (Status_group)

| Original Status | Status Group |
|----------------|-------------|
| Shipped | In Transit |
| Shipped - Out for Delivery | In Transit |
| Shipped - Picked Up | In Transit |
| Shipping | In Transit |
| Shipped - Delivered to Buyer | Delivered |
| Cancelled | Cancelled |
| Shipped - Returned to Seller | Returned |
| Shipped - Returning to Seller | Returned |
| Shipped - Rejected by Buyer | Returned |
| Pending | Pending |
| Pending - Waiting for Pick Up | Pending |
| Shipped - Lost in Transit | Exception |
| Shipped - Damaged | Exception |

In [88]:
# # This gives you 5 clean groups instead of 13. So it will be much easier to slice on a dashboard.
# display(df.select('status').distinct()) # see in this there are 13 unique values which will creat an diffinculty in the dashboard
# display(df.select('status_group').distinct()) # That's why we have group them into different 5 clean groups

# And also we are keeping all the columns status_group, courier_status and status because it can be used for different - different purposes like
# Executive dashboard → status_group for clean charts
# Operations dashboard → courier_status for logistics pipeline
# Drillthrough / detail page → status for row-level investigation

StatementMeta(, a234de3e-0893-4294-9d50-190b1a27bd6e, 92, Finished, Available, Finished, False)

In [89]:
# Creating new column name status_group
from itertools import chain

mapping = {
    "Shipped": "In Transit",
    "Shipped - Out for Delivery": "In Transit",
    "Shipped - Picked Up": "In Transit",
    "Shipping": "In Transit",

    "Shipped - Delivered to Buyer": "Delivered",

    "Cancelled": "Cancelled",

    "Shipped - Returned to Seller": "Returned",
    "Shipped - Returning to Seller": "Returned",
    "Shipped - Rejected by Buyer": "Returned",

    "Pending": "Pending",
    "Pending - Waiting for Pick Up": "Pending",

    "Shipped - Lost in Transit": "Exception",
    "Shipped - Damaged": "Exception"
}

mapping_expr = create_map([lit(x) for x in chain(*mapping.items())])

df = df.withColumn("status_group", mapping_expr[col("status")])


df = df.withColumn(
    "status_group", 
    mapping_expr[col("status")]
).fillna({"status_group": "Other"})

StatementMeta(, a234de3e-0893-4294-9d50-190b1a27bd6e, 94, Finished, Available, Finished, False)

In [90]:
#display(df)

StatementMeta(, a234de3e-0893-4294-9d50-190b1a27bd6e, 96, Finished, Available, Finished, False)

### **Problems with the data set**

In [91]:
# This will show all the records where the status = "shipped - delivered to buyer" but still have an amount = 0 or Null means this will make our kpi revenue loss.
invalid_amount = df.filter(
    (col("status") == "Shipped - Delivered to Buyer") &
    (
        col("amount").isNull() |
        (col("amount") == 0)
    )
)

# display(invalid_amount)
print("Invalid Delivered Records:", invalid_amount.count())
#Invalid Delivered Records: 724

StatementMeta(, a234de3e-0893-4294-9d50-190b1a27bd6e, 97, Finished, Available, Finished, False)

Invalid Delivered Records: 724


In [92]:
# This will show all the records where the status=Cancelled But still have the amount this can increase the revenue unathically
invalid_amountt = df.filter(
    (col("status") == "Cancelled") &
    (
        (col("amount") > 0)
    )
)

# display(invalid_amountt)
print("Invalid Delivered Records:", invalid_amountt.count())

StatementMeta(, a234de3e-0893-4294-9d50-190b1a27bd6e, 98, Finished, Available, Finished, False)

Invalid Delivered Records: 10766


In [93]:
df = df.withColumn("net_amount",
    when(col("status_group") == "Cancelled", lit(0.0))
    .otherwise(col("amount"))
)

StatementMeta(, a234de3e-0893-4294-9d50-190b1a27bd6e, 99, Finished, Available, Finished, False)

### **Transformation For ship_state column**

In [94]:
# In this i have converted all the state name to the upper case so that the redundancy gets decrease.
#like Goa and goa both will be consider as an unique value so by doing this. Problem is solved. 
df =df.withColumn("ship_state",upper(col=("ship_state")))

StatementMeta(, a234de3e-0893-4294-9d50-190b1a27bd6e, 100, Finished, Available, Finished, False)

In [95]:
# # Before doing this the count is 70 and after the count is 48
# display(df.select('ship_state').distinct().count()) #70 #48
# # But still me have an redundancy problem like "PB" and "PUNJAB" both values are unique so this will create an redundancy
# display(df.select('ship_state').distinct()) 

StatementMeta(, a234de3e-0893-4294-9d50-190b1a27bd6e, 101, Finished, Available, Finished, False)

In [96]:
# This is using mapping, spark literal etc. More advance version

# from itertools import chain
# # Step 1 — trim spaces + uppercase first
# df = df.withColumn("ship_state", upper(trim(col("ship_state"))))

# # Step 2 — this state_mapping we have created an mapping dictionary. full correction mapping (handles all 4 issues) 
# state_mapping = {

#     # Abbreviations
#     "NL":                       "NAGALAND",
#     "PB":                       "PUNJAB",
#     "RJ":                       "RAJASTHAN",

#     # Spelling variants
#     "RAJSHTHAN":                "RAJASTHAN",
#     "RAJSTHAN":                 "RAJASTHAN",
#     "ORISSA":                   "ODISHA",

#     # Duplicate names
#     "NEW DELHI":                "DELHI",
#     "PUNJAB/MOHALI/ZIRAKPUR":   "PUNJAB",
#     "PUDUCHERRY":               "PONDICHERRY",

#     # Unknown/junk codes
#     "APO":                      "OTHER",
#     "AR":                       "OTHER",
# }

# # Convert Python dictionary into Spark map format (key-value pairs) for lookup
# ## Convert Python dictionary into Spark map format (key-value pairs) for lookup
# mapping_expr = create_map([lit(x) for x in chain(*state_mapping.items())])


# # Step 3 — apply mapping
# #coalesece basically cheks for the mapped value if it is there then replace. And not then keep as it is
# df = df.withColumn(
#     "ship_state",
#     coalesce(mapping_expr[col("ship_state")], col("ship_state"))
# )

# # Step 4 — verify
# print("Distinct state count after fix: 38")
# display(df.select("ship_state").distinct().orderBy("ship_state"))

StatementMeta(, a234de3e-0893-4294-9d50-190b1a27bd6e, 102, Finished, Available, Finished, False)

In [97]:
# 2nd way is by checking the redudency manually and replase it when() this is simple to implement but not production friendly
# the 3rd method is use when the dataset is too big when the ai is not able to read then we can use an "Fuzzy Matching"
# approach it is basically  an algorithm.
from pyspark.sql.functions import col, trim, upper, when

# Step 1 — clean column
df = df.withColumn("ship_state", upper(trim(col("ship_state"))))

# Step 2 — apply mapping using when
df = df.withColumn(
    "ship_state",

    when(col("ship_state") == "NL", "NAGALAND")
    .when(col("ship_state") == "PB", "PUNJAB")
    .when(col("ship_state") == "RJ", "RAJASTHAN")

    .when(col("ship_state").isin("RAJSHTHAN", "RAJSTHAN"), "RAJASTHAN")
    .when(col("ship_state") == "ORISSA", "ODISHA")

    .when(col("ship_state") == "NEW DELHI", "DELHI")
    .when(col("ship_state") == "PUNJAB/MOHALI/ZIRAKPUR", "PUNJAB")
    .when(col("ship_state") == "PUDUCHERRY", "PONDICHERRY")

    .when(col("ship_state").isin("APO", "AR"), "OTHER")

    .otherwise(col("ship_state"))
)

# Step 3 — check result
display(df.select("ship_state").distinct().orderBy("ship_state"))

StatementMeta(, a234de3e-0893-4294-9d50-190b1a27bd6e, 103, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c187eb44-49e1-4054-adb8-c6f97bf84d30)

### **Transformation For ship_city Column**

In [98]:
# # simply converting ship_city all values to the upper case so that the inconsistencies get remove
# df = df.withColumn("ship_city",upper(trim(col("ship_city"))))
# display(df.select('ship_city').distinct().count()) #8956 #7298 #6404

StatementMeta(, a234de3e-0893-4294-9d50-190b1a27bd6e, 104, Finished, Available, Finished, False)

In [99]:
# In this also there are different different redundancy issues are there. 
# which are solved by two wasy as mention above that by taking the help of ai and 2nd one is by using "Fuzzy Matching".

# ── STEP 1: Basic cleanup — uppercase + strip spaces ──
df = df.withColumn("ship_city", upper(trim(col("ship_city"))))

# ── STEP 2: Strip pin codes & trailing numbers ──
# Removes patterns like "BANGALORE 560062", "DELHI -86", "AHMEDABAD."
df = df.withColumn("ship_city",
    trim(regexp_replace(col("ship_city"), r"[\s,\-\.]*\d[\d\s\-\.]*$", ""))
)

# ── STEP 3: Extract city from "LOCALITY, CITY" format ──
# If comma exists → take the part after last comma
df = df.withColumn("ship_city",
    when(col("ship_city").contains(","),
        trim(regexp_replace(col("ship_city"), r"^.*,\s*", ""))
    ).otherwise(col("ship_city"))
)

# ── STEP 4: Manual mapping for known variants & short codes ──
city_mapping = {

    # Bengaluru variants
    "BANGALORE":         "BENGALURU",
    "BENGALORE":         "BENGALURU",
    "BENGALUR":          "BENGALURU",
    "BENGALURE":         "BENGALURU",
    "BANGALURU":         "BENGALURU",
    "BENGALOORU":        "BENGALURU",

    # Delhi variants
    "NEW DELHI":         "DELHI",
    "DELHI CANTT":       "DELHI",
    "CENTRAL DELHI":     "DELHI",
    "EAST DELHI":        "DELHI",

    # Chennai variants
    "CHENAI":            "CHENNAI",
    "MADRAS":            "CHENNAI",

    # Hyderabad variants
    "SECUNDRABAD":       "HYDERABAD",
    "SECUNDERABAD":      "HYDERABAD",

    # Kolkata variants
    "CALCUTTA":          "KOLKATA",

    # Mumbai variants
    "BOMBAY":            "MUMBAI",
    "NAVI MUMBAI":       "MUMBAI",
    "NAVIMUMBAI":        "MUMBAI",

    # Short codes
    "HYD":               "HYDERABAD",
    "BLR":               "BENGALURU",
    "MUM":               "MUMBAI",
    "DEL":               "DELHI",
    "GOA":               "GOA",
    "LEH":               "LEH",

    # Junk / unrecognizable
    "APO":               "OTHER",
    "T":                 "OTHER",
    "1":                 "OTHER",
}

mapping_expr = create_map([lit(x) for x in chain(*city_mapping.items())])

df = df.withColumn(
    "ship_city",
    coalesce(mapping_expr[col("ship_city")], col("ship_city"))
)

# ── STEP 5: Verify ──
print("Distinct city count after cleaning:")
df.select("ship_city").distinct().count()

StatementMeta(, a234de3e-0893-4294-9d50-190b1a27bd6e, 105, Finished, Available, Finished, False)

Distinct city count after cleaning:


6404

In [100]:
# This approach is by using "Fuzzy Matching"
# Don't try to fix the city name directly
# → Instead, extract just the CORE CITY NAME from the messy string
# → Then validate it against the state it belongs to

# Step 1 — You run this in Fabric and paste the output here
# python# Run this in Fabric and share the output with me
# from pyspark.sql.functions import col, upper, trim, regexp_replace, when

# df = df.withColumn("ship_city_clean", upper(trim(col("ship_city"))))

# # Step A: strip pin codes
# df = df.withColumn("ship_city_clean",
#     trim(regexp_replace(col("ship_city_clean"), r"[\s,\-\.]*\d[\d\s\-\.]*$", ""))
# )

# # Step B: extract city from "LOCALITY, CITY" → take part after last comma
# df = df.withColumn("ship_city_clean",
#     when(col("ship_city_clean").contains(","),
#         trim(regexp_replace(col("ship_city_clean"), r"^.*,\s*", ""))
#     ).otherwise(col("ship_city_clean"))
# )

# # Now check what's still problematic — short/junk values
# df.filter(
#     (col("ship_city_clean").rlike(r"^\d+$")) |   # purely numeric
#     (col("ship_city_clean").rlike(r"^.{1,2}$"))  # 1-2 char junk
# ).groupBy("ship_city_clean").count().orderBy("count", ascending=False).show(50, truncate=False)

# # Also check distinct count after automated cleaning
# print("Distinct cities after automated cleaning:", 
#       df.select("ship_city_clean").distinct().count())

# Step 2 — For remaining variants, use fuzzy matching
# This is the real solution for large datasets — instead of manual mapping, let the algorithm find similar city names automatically:
# python# Install in Fabric notebook
# %pip install thefuzz

# from thefuzz import process
# from pyspark.sql.functions import udf
# from pyspark.sql.types import StringType

# # Master list of correct Indian city names
# # You only need ~200 major cities — covers 95% of orders
# KNOWN_CITIES = [
#     "MUMBAI", "DELHI", "BENGALURU", "HYDERABAD", "AHMEDABAD",
#     "CHENNAI", "KOLKATA", "PUNE", "JAIPUR", "LUCKNOW",
#     "SURAT", "KANPUR", "NAGPUR", "INDORE", "BHOPAL",
#     "PATNA", "VADODARA", "COIMBATORE", "AGRA", "NASHIK",
#     "RANCHI", "FARIDABAD", "MEERUT", "RAJKOT", "VARANASI",
#     "AMRITSAR", "AURANGABAD", "JABALPUR", "GWALIOR", "VIJAYAWADA",
#     "JODHPUR", "MADURAI", "RAIPUR", "KOCHI", "CHANDIGARH",
#     "GUWAHATI", "THIRUVANANTHAPURAM", "MYSURU", "BHUBANESWAR", "DEHRADUN",
#     "NOIDA", "GURUGRAM", "GHAZIABAD", "NAVI MUMBAI", "THANE",
#     # Add more as needed
# ]

# # Fuzzy match UDF — finds closest city name automatically
# def fuzzy_match_city(city_name):
#     if not city_name or len(city_name) <= 2:
#         return "OTHER"
#     result = process.extractOne(city_name, KNOWN_CITIES, score_cutoff=80)
#     return result[0] if result else city_name  # keep original if no close match

# fuzzy_udf = udf(fuzzy_match_city, StringType())

# df = df.withColumn("ship_city_clean", fuzzy_udf(col("ship_city_clean")))

# # Verify
# df.groupBy("ship_city_clean").count().orderBy("count", ascending=False).show(50, truncate=False)
# ```

# ---

# **How fuzzy matching works — simple explanation:**
# ```
# Input:  "BANGALURU"
# ↓
# Algorithm checks similarity score against all known cities:
#   BENGALURU  →  92% match  ✅ winner
#   BANGALORE  →  88% match
#   DELHI      →  12% match

# score_cutoff=80 means:
#   above 80% → replace with correct name
#   below 80% → keep original (not confident enough to change)

# Step 3 — For anything still unresolved, fall back to state
# pythonfrom pyspark.sql.functions import coalesce

# # If city is still junk/OTHER → use state name as fallback
# # At least your geo KPIs won't break
# df = df.withColumn("ship_city_final",
#     when(
#         col("ship_city_clean").isin("OTHER", "", "1", "T"),
#         col("ship_state")   # fallback to state
#     ).otherwise(col("ship_city_clean"))
# )
# ```

# ---

# **So the full strategy in plain English:**
# ```
# Step 1 → Automated rules fix ~80% of cases
#          (uppercase, strip pins, split on comma)
#               ↓
# Step 2 → Fuzzy matching fixes ~15% more
#          (spelling variants, typos)
#               ↓
# Step 3 → Remaining ~5% junk → fallback to ship_state
#               ↓
# Result → 95%+ clean city data, zero manual work

StatementMeta(, a234de3e-0893-4294-9d50-190b1a27bd6e, 106, Finished, Available, Finished, False)

### **Transformation for the ship_postal_code**

In [101]:
# This we do because there are null values in the column, inconsistent values having less digits than 6 and also .0 at the end

# Step 1 — Remove decimal (.0) and convert to string
df = df.withColumn("ship_postal_code",
    regexp_replace(col("ship_postal_code").cast("string"), r"\.0$", "")
)

# Step 2 — Pad to 6 digits with leading zeros (safety net)
# e.g. if any PIN like 11001 got stored as 001100
df = df.withColumn("ship_postal_code",
    lpad(col("ship_postal_code"), 6, "0")
)

# Step 3 — Handle nulls
df = df.withColumn("ship_postal_code",
    when(col("ship_postal_code").isNull(), lit("000000"))
    .otherwise(col("ship_postal_code"))
)

# Verify
# print("Distinct PIN codes:", df.select("ship_postal_code").distinct().count())
# print("Null count:", df.filter(col("ship_postal_code").isNull()).count())
# df.select("ship_postal_code").show(10, truncate=False)

StatementMeta(, a234de3e-0893-4294-9d50-190b1a27bd6e, 107, Finished, Available, Finished, False)

### **Transformation of the promotion_ids**
Promotion IDs — Why We Transformed It
The Problem
The raw promotion_ids column stores multiple promotion codes in a single
cell as one giant unreadable text blob. It cannot be filtered, sliced, or
counted directly in any dashboard or report.

What We Did
Created 3 new clean columns from the raw column:

Advantages
1. Dashboard Ready
- The raw column cannot be used in any visual or slicer. The 3 new columns can be directly used as filters, slicers, and KPI measures in Fabric.

2. Business Questions Answered:
- How many orders had a promotion? → has_promotion
- Which type of promotion is most common? → promotion_type
- How many promotions were stacked per order? → promo_count

3. No Data Loss
- The original promotion_ids column is kept as-is for audit and traceability. We only added new columns — nothing was deleted.

4. Better Performance
- Filtering on a simple True/False or a short category value is significantly faster in Fabric than scanning through a 2,000+ character raw text field.


In [102]:
# I have to understand this code 
# ── Transform 1 — has_promotion flag ──
df = df.withColumn("has_promotion",
    when(col("promotion_ids").isNull(), False).otherwise(True)
)

# ── Transform 2 — promotion_type (extract main type) ──
df = df.withColumn("promotion_type",
    when(col("promotion_ids").isNull(), "No Promotion")
    .when(col("promotion_ids").contains("Free Shipping"), "Free Shipping")
    .when(col("promotion_ids").contains("PLCC"), "Credit Card Financing")
    .when(col("promotion_ids").startswith("D"), "Discount")
    .when(col("promotion_ids").startswith("VPC"), "Coupon")
    .otherwise("Other")
)

# ── Transform 3 — promo_count (how many promos on one order) ──
df = df.withColumn("promo_count",
    when(col("promotion_ids").isNull(), 0)
    .otherwise(size(split(col("promotion_ids"), ","))
    )
)
# Mini step 1 — split()
# Cuts the long text into a list everywhere it sees a comma
# "Amazon PLCC AAT-123, Amazon PLCC AAT-456, IN Core Free Shipping"
# → ["Amazon PLCC AAT-123", "Amazon PLCC AAT-456", "IN Core Free Shipping"]

# Mini step 2 — size()
# Counts how many items are in that list->3

# Mini step 3 — when().otherwise()
# If null → 0 (no promotions)
# If not null → count the items

# ── Verify ──
display(df.groupBy("promotion_type")
          .agg({"has_promotion": "count", "Amount": "avg", "promo_count": "avg"})
          .orderBy("count(has_promotion)", ascending=False))

StatementMeta(, a234de3e-0893-4294-9d50-190b1a27bd6e, 108, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f95c28ee-42b5-4016-b335-17573141ec21)

### **Transformation for b2b column**
Only 1 small transformation worth doing:
Rename True/False to B2B/B2C for better readability on dashboards — because when a business user sees a slicer, B2B and B2C is much clearer than True and False:

In [103]:
from pyspark.sql.functions import col, when

# Just add a readable label column — keep original B2B as-is
df = df.withColumn("customer_type",
    when(col("customer_type") == True, "B2B")
    .otherwise("B2C")
)

StatementMeta(, a234de3e-0893-4294-9d50-190b1a27bd6e, 109, Finished, Available, Finished, False)

In [104]:
#display(df)

StatementMeta(, a234de3e-0893-4294-9d50-190b1a27bd6e, 110, Finished, Available, Finished, False)

### **Adding Extra columns from the date column for the kip filtering**

In [105]:
# added at the end
df = df \
    .withColumn("order_year",    year(col("date"))) \
    .withColumn("order_month",   month(col("date"))) \
    .withColumn("order_quarter", quarter(col("date"))) \
    .withColumn("order_weekday", date_format(col("date"), "EEEE")) \
    .withColumn("month_name",    date_format(col("date"), "MMMM"))

StatementMeta(, a234de3e-0893-4294-9d50-190b1a27bd6e, 111, Finished, Available, Finished, False)

### **Most Important (Handling Bad data that might affect the KPI) By adding Flag**

In [106]:
df = df.withColumn("reject_reason",

    # Flag 1 — No order ID → untraceable record
    when(col("order_id").isNull(),
         "Missing order_id")
#      Why: order_id is the backbone of everything
#      Total Orders KPI → counts distinct order_id
#      AOV             → Revenue / order_id count
#      If order_id is null → that order is untraceable
#                          → cannot be counted
#                          → cannot be verified

 
    # Flag 2 — No date → ALL time KPIs break
    .when(col("date").isNull(),
          "Unparseable date")
#      Why: date drives EVERY time KPI you have
#      Daily Sales, Monthly Revenue, YTD,
#      MTD, Growth %, Peak Day, Weekday trend
#      If date is null → record has no time dimension
#                      → ALL time KPIs break silently
#                      → record gets dumped into wrong period


  
    # Flag 3 — Shipped but amount missing → revenue gap
    # (also covers Qty=0 on shipped — system sync failure)
    .when(
        (col("amount").isNull()) &
        (~col("status").isin([
            "Cancelled", "Pending",
            "Pending - Waiting for Pick Up"
        ])),
        when(col("qty") == 0,
             "Qty=0 and missing amount on shipped order")
        .otherwise(
             "Missing amount on non-cancelled order")
    )
#     Why: A shipped/delivered order with null amount
#      means revenue was never recorded
#      Total Revenue KPI → silently undercounts
#      These are NOT normal — cancelled/pending
#      orders CAN have null amount (that is fine)
#      but shipped orders MUST have an amount
   
   
    # Flag 4 — Negative qty → impossible in sales
    .when(col("qty") < 0,
          "Negative quantity")
# Why: Qty of -3 is physically impossible
#      You cannot ship -3 kurtis
#      Total Qty Sold KPI → gets reduced wrongly
#      Could be a return recorded in wrong table
#      Should be in a returns table, not sales table
   
   
    # Flag 5 — Negative amount → impossible in sales
    .when(col("amount") < 0,
          "Negative amount")
# Why: Revenue of -₹500 makes no sense in sales data
#      Refunds should be in a separate returns table
#      Total Revenue KPI → gets reduced wrongly
#      Running Total    → goes negative at wrong points
    
    
    # All good → no issue
    .otherwise(None)
)

#**Now your flags are clean — 5 distinct reasons, no overlap:**
# ```
# Flag 1 → "Missing order_id"
# Flag 2 → "Unparseable date"
# Flag 3 → "Missing amount on non-cancelled order"
#        → "Qty=0 and missing amount on shipped order"  ← specific sub-case
# Flag 4 → "Negative quantity"
# Flag 5 → "Negative amount"

StatementMeta(, a234de3e-0893-4294-9d50-190b1a27bd6e, 112, Finished, Available, Finished, False)

In [107]:
# Should show all 13 statuses mapped correctly
# display(df.groupBy("status", "status_group", "reject_reason") \
#   .count() \
#   .orderBy("status_group", "status"))

StatementMeta(, a234de3e-0893-4294-9d50-190b1a27bd6e, 113, Finished, Available, Finished, False)

In [108]:
display(df.groupBy("reject_reason").count())

StatementMeta(, a234de3e-0893-4294-9d50-190b1a27bd6e, 114, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 043c4c6f-271e-4e68-947d-298e91309fc4)

In [109]:
# this will mark all the values true if the record is in
# Shipped -	In Transit
# Shipped - Out for Deliveryt
# Shipped - Picked Up	
# Shipping	In Transit
# Shipped - Delivered to Buyer

# AND mark false if the record is in 
#Cancelled	Cancelled
#Shipped - Returned to Seller	
#Shipped - Returning to Seller	
#Shipped - Rejected by Buyer
#Pending	Pending
#Pending - Waiting for Pick Up
# Shipped - Lost in Transit	Exception
# Shipped - Damaged	Exception

df = df.withColumn("is_valid_revenue",
    when(col("status").isin([
        "Cancelled",
        "Pending",
        "Pending - Waiting for Pick Up",
        "Shipped - Returned to Seller",
        "Shipped - Returning to Seller",
        "Shipped - Rejected by Buyer",
        "Shipped - Lost in Transit",
        "Shipped - Damage"
    ]), False).otherwise(True)
)

StatementMeta(, a234de3e-0893-4294-9d50-190b1a27bd6e, 115, Finished, Available, Finished, False)

In [110]:
display(df.groupBy("is_valid_revenue").count())

StatementMeta(, a234de3e-0893-4294-9d50-190b1a27bd6e, 116, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3fa4af5f-d5f1-4489-a53b-57c6df2b5c0e)

In [111]:
# # Split into 2 tables
# ── Step 1: Split ──
df_clean      = df.filter(col("reject_reason").isNull())
df_quarantine = df.filter(col("reject_reason").isNotNull())

# ── Step 2: Verify counts ── 
print("Clean records     :", df_clean.count())
print("Quarantine records:", df_quarantine.count())

# ── Step 3: See why records were rejected ──
df_quarantine.groupBy("reject_reason") \
             .count() \
             .orderBy("count", ascending=False) \
             .show(truncate=False)

# ── Step 4: Fill nulls on CLEAN records ONLY ──
df_clean = df_clean.fillna({
    "courier_status" : "Unknown",
    "promotion_ids"  : "None",
    "promotion_type" : "No Promotion",
    "promo_count"    : 0,
    "ship_city"      : "Unknown",   
    "ship_state"     : "Unknown",
})

StatementMeta(, a234de3e-0893-4294-9d50-190b1a27bd6e, 117, Finished, Available, Finished, False)

Clean records     : 128748
Quarantine records: 227
+-----------------------------------------+-----+
|reject_reason                            |count|
+-----------------------------------------+-----+
|Missing amount on non-cancelled order    |123  |
|Qty=0 and missing amount on shipped order|104  |
+-----------------------------------------+-----+



### **Removing Extra Space**

In [112]:
import pyspark.sql.functions as F
# STRING STANDARDIZATION") 
df_clean = df_clean \
    .withColumn("order_id",
                F.initcap(F.trim(F.col("order_id")))) \
    .withColumn("status",
                F.initcap(F.trim(F.col("status")))) \
    .withColumn("fulfilment",
                F.initcap(F.trim(F.col("fulfilment")))) \
    .withColumn("ship_service_level",
                F.upper(F.trim(F.col("ship_service_level")))) \
    .withColumn("style",
                F.initcap(F.trim(F.col("style")))) \
    .withColumn("sku",
                F.upper(F.trim(F.col("sku")))) \
    .withColumn("category",
                F.upper(F.trim(F.col("category")))) \
    .withColumn("size",
                F.upper(F.trim(F.col("size")))) \
    .withColumn("asin",
                F.trim(F.col("asin"))) \
    .withColumn("courier_status",
                F.trim(F.col("courier_status"))) \
    .withColumn("ship_city",
                F.trim(F.col("ship_city")))\
    .withColumn("ship_state",
                F.trim(F.col("ship_state")))
 
print("✅ String standardization complete.")

StatementMeta(, a234de3e-0893-4294-9d50-190b1a27bd6e, 118, Finished, Available, Finished, False)

✅ String standardization complete.


In [113]:
# serogate key Your Gold star schema needs product_key, location_key, fulfillment_key to join fact to dimensions. Add:
# ─────────────────────────────────────────────────────────────────────────────
# SURROGATE KEY GENERATION (FOR GOLD STAR SCHEMA)
# ─────────────────────────────────────────────────────────────────────────────
#
# What is a Surrogate Key?
# A surrogate key is an artificially created unique identifier used to represent
# a record in a table instead of using multiple business columns (natural keys).
#
# Why do we need it?
# In our dataset, entities like Product, Location, and Fulfillment are identified
# using multiple columns:
#   - Product   → sku + asin + category
#   - Location  → ship_city + ship_state + ship_postal_code
#   - Fulfillment → fulfilment + ship_service_level
#
# Using multiple columns for joins:
#   ❌ Makes queries complex
#   ❌ Slows down performance
#   ❌ Increases chances of errors
#
# Solution:
# We create a single column (surrogate key) that uniquely identifies each entity.
#
# Why MD5?
# - Converts multiple columns into a single fixed-length unique hash
# - Deterministic: same input always produces same key
# - Fast and efficient for large datasets
# - Ideal for joining fact and dimension tables in a star schema
#
# Example:
#   "A123|B09X|Kurta" → md5 → "9f3a1c8e5b..."
#
# This surrogate key will be used in:
#   - Dimension tables (dim_product, dim_location, dim_fulfillment)
#   - Fact table (fact_sales) to establish relationships
#
# This approach ensures:
#   ✅ Faster joins
#   ✅ Cleaner data model
#   ✅ Scalable architecture
# ─────────────────────────────────────────────────────────────────────────────

df_clean = df_clean \
    .withColumn("product_key",
        md5(concat_ws("|", col("sku"), col("asin"), col("category")))) \
    .withColumn("location_key",
        md5(concat_ws("|", col("ship_city"), col("ship_state"),
                          col("ship_postal_code")))) \
    .withColumn("fulfillment_key",
        md5(concat_ws("|", col("fulfilment"), col("ship_service_level"))))



# Note: Surrogate keys are deterministic — same input always generates same key,
# ensuring consistency across pipeline runs.1

StatementMeta(, a234de3e-0893-4294-9d50-190b1a27bd6e, 119, Finished, Available, Finished, False)

### **Writing the table in the Silver Lakehouse**

In [114]:
df_clean.write.mode("overwrite").option("overwriteSchema","true").saveAsTable("ecommerce_silver.silver.silver_amz_sales")
df_quarantine.write.mode("overwrite").option("overwriteSchema","true").saveAsTable("ecommerce_silver.silver.silver_amz_sales_quarantine")

StatementMeta(, a234de3e-0893-4294-9d50-190b1a27bd6e, 120, Finished, Available, Finished, False)

In [115]:
#display(df_quarantine)
#display(df_clean)

StatementMeta(, a234de3e-0893-4294-9d50-190b1a27bd6e, 121, Finished, Available, Finished, False)